# ML Models
Exécution des 3 modèles ML : BERTopic (topic modeling), NER (entités), Isolation Forest (anomalies).

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

project_root = Path.cwd()
if not (project_root / 'data').exists():
    project_root = project_root.parent
sys.path.append(str(project_root))

data_path = project_root / 'data/processed/gdelt_benin_clean.csv'
df = pd.read_csv(data_path)
df['date_parsed'] = pd.to_datetime(df['date'], errors='coerce')
print(f'Loaded {len(df)} rows from {data_path.name}')
print(f'Date range: {df["date_parsed"].min()} → {df["date_parsed"].max()}')

: 

## Model 1: BERTopic (Topic Modeling)
Identifie les thèmes médiatiques récurrents autour du Bénin.

In [ ]:
from src.models.topic_model import extract_topics

benin_mask = (
    df.get('Actor1CountryCode', pd.Series(dtype=str)).astype(str).eq('BEN') |
    df.get('ActionGeo_FullName', pd.Series(dtype=str)).astype(str).str.contains('benin', case=False, na=False)
 )
raw_texts = df.loc[benin_mask, 'event_label'].astype(str).dropna()
benin_texts = [t.strip() for t in raw_texts if isinstance(t, str) and t.strip() and len(t.strip()) > 10]
if len(benin_texts) == 0:
    print('No Benin-related event_label texts found — skipping BERTopic.')
    topic_model = None
    topic_info = pd.DataFrame(columns=['Topic', 'Count'])
else:
    sample_size = min(300, len(benin_texts))
    sample_texts = np.random.choice(benin_texts, sample_size, replace=False).tolist()
    sample_texts = [s for s in sample_texts if s and len(s.strip()) > 5]
    if len(sample_texts) < 20:
        print(f'Too few samples ({len(sample_texts)}) — skipping BERTopic.')
        topic_model = None
        topic_info = pd.DataFrame(columns=['Topic', 'Count'])
    else:
        print(f'Training BERTopic on {len(sample_texts)} samples...')
        result = extract_topics(sample_texts, min_topic_size=15)

if 'result' in locals():
    topic_model = result.model
    topics_list = result.topics
    topic_info = topic_model.get_topic_info()
    print(f'\nTopics found: {len(topic_info)}')
    print('\nTop 5 topics by frequency:')
    print(topic_info.head(5)[['Topic', 'Count']].to_string(index=False))
else:
    topic_model = None
    topics_list = []
    topic_info = pd.DataFrame(columns=['Topic', 'Count'])

Training BERTopic on 300 unique Benin-related events...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2909.45it/s]



Topics found: 7

Top 5 topics by frequency:
 Topic  Count
     0     82
     1     64
     2     56
     3     30
     4     28


In [10]:
print('\nTopic Keywords:')
for topic_id in topic_info['Topic'].head(5):
    if topic_id == -1:
        print(f'Topic {topic_id}: [Outliers]')
        continue
    keywords = topic_model.get_topic(topic_id)
    kw_str = ', '.join([kw for kw, _ in keywords[:8]])
    print(f'Topic {topic_id}: {kw_str}')


Topic Keywords:
Topic 0: publique, déclaration, protestation, , , , , 
Topic 1: consultation, , , , , , , 
Topic 2: diplomatique, engagement, force, militaire, non, , , 
Topic 3: désapprobation, refus, rejet, , , , , 
Topic 4: demande, appel, aide, assistance, , , , 


## Model 2: NER (Named Entity Recognition)
Extrait les entités nommées : personnes, organisations, lieux.

In [7]:
from src.models.ner_model import extract_entities
from collections import Counter
import re

# Prepare text for NER
df['ner_text'] = (
    df['Actor1Name'].fillna('').astype(str) + ' ' +
    df['Actor2Name'].fillna('').astype(str) + ' ' +
    df['ActionGeo_FullName'].fillna('').astype(str)
).str.strip()

print('Extracting entities...')
entities = extract_entities(df['ner_text'].tolist())

# Aggregate
person_counter = Counter()
org_counter = Counter()
location_counter = Counter()

NOISE_TOKENS = {'INCONNU', 'UNKNOWN', 'NA', 'N/A', 'NONE', 'NULL'}
COUNTRY_NOISE = {'BENIN', 'BEN', 'BENINESE'}

def clean_entity(val, kind):
    text = re.sub(r'\s+', ' ', str(val or '')).strip()
    if not text:
        return ''
    parts = [p.strip() for p in re.split(r'[\s,;/|]+', text) if p.strip()]
    cleaned = []
    for p in parts:
        upper_p = p.upper()
        if upper_p in NOISE_TOKENS:
            continue
        if kind in ('persons', 'orgs') and upper_p in COUNTRY_NOISE:
            continue
        cleaned.append(p.title() if p.isupper() else p)
    result = ' '.join(cleaned).strip()
    return result if len(result) >= 3 and not result.isdigit() else ''

for entity_dict in entities:
    for v in entity_dict.get('persons', []):
        clean_v = clean_entity(v, 'persons')
        if clean_v:
            person_counter[clean_v] += 1
    for v in entity_dict.get('orgs', []):
        clean_v = clean_entity(v, 'orgs')
        if clean_v:
            org_counter[clean_v] += 1
    for v in entity_dict.get('locations', []):
        clean_v = clean_entity(v, 'locations')
        if clean_v:
            location_counter[clean_v] += 1

print(f'\nExtracted {len(person_counter)} persons, {len(org_counter)} orgs, {len(location_counter)} locations')
print('\nTop 10 Persons:')
for person, count in person_counter.most_common(10):
    print(f'  {person}: {count}')
print('\nTop 10 Organizations:')
for org, count in org_counter.most_common(10):
    print(f'  {org}: {count}')
print('\nTop 10 Locations:')
for loc, count in location_counter.most_common(10):
    print(f'  {loc}: {count}')

Extracting entities...

Extracted 99 persons, 491 orgs, 19 locations

Top 10 Persons:
  Atlantique: 120
  Zou: 64
  Qué: 63
  Porto-Novo: 30
  Ghana: 20
  Mono: 17
  Kandi: 13
  Alibori: 12
  Senegal: 10
  Borgou: 9

Top 10 Organizations:
  Nigeria: 261
  Government: 174
  Africa: 148
  Governor: 138
  Police: 106
  President: 101
  Alibori: 94
  Ouidah: 61
  Nigerian: 59
  Military: 52

Top 10 Locations:
  Benin: 270
  Borgou: 63
  Donga: 13
  Parakou: 12
  Atakora: 9
  Porga: 8
  Kouffo: 8
  Malanville: 8
  Kandi: 2
  Segbana: 2


## Model 3: Isolation Forest (Anomaly Detection)
Détecte les mois avec patterns anormaux (pics de volume, sentiment ou activité géopolitique).

In [8]:
from src.models.anomaly_model import detect_anomalies
from src.models.baseline_anomaly import detect_anomalies_iqr

ml_monthly = (
    df.assign(month=df['date_parsed'].dt.to_period('M'))
      .groupby('month', as_index=False)
      .agg(
          rows=('event_label', 'size'),
          avg_tone=('AvgTone', 'mean'),
          goldstein_scale=('GoldsteinScale', 'mean'),
          num_articles=('NumArticles', 'sum'),
      )
      .dropna()
      .sort_values('month')
      .reset_index(drop=True)
)

print(f'Monthly aggregated data: {len(ml_monthly)} months')
print('\nMonthly stats:')
print(ml_monthly[['month', 'rows', 'avg_tone', 'goldstein_scale']].head(10).to_string(index=False))

Monthly aggregated data: 16 months

Monthly stats:
  month  rows  avg_tone  goldstein_scale
2025-01   532 -1.213568         0.418985
2025-02   363 -1.581355         0.281543
2025-03   442 -1.767629         0.481448
2025-04   421 -1.031985         0.576960
2025-05   426 -0.019950         0.903286
2025-06   209 -1.575583         0.703828
2025-07   465 -0.392100         1.015914
2025-08   324 -0.966941         0.687654
2025-09   330 -0.505516         0.745152
2025-10   373  0.486699         1.248257


In [11]:
# Run Isolation Forest
print('Running Isolation Forest...')
anomaly_result = detect_anomalies(
    dataframe=ml_monthly,
    feature_columns=['rows', 'avg_tone', 'goldstein_scale', 'num_articles'],
    contamination=0.1,
    random_state=42,
)

# Run IQR baseline
baseline_result = detect_anomalies_iqr(
    dataframe=ml_monthly,
    feature_columns=['rows', 'avg_tone', 'goldstein_scale', 'num_articles'],
)

if_anomalies = anomaly_result.dataframe[anomaly_result.dataframe['is_anomaly']]
iqr_anomalies = baseline_result.dataframe[baseline_result.dataframe['iqr_is_anomaly']]

print(f'\nIsolation Forest detected {len(if_anomalies)} anomalous months:')
if len(if_anomalies) > 0:
    print(if_anomalies[['month', 'rows', 'anomaly_score']].sort_values('anomaly_score', ascending=False).to_string(index=False))

print(f'\nIQR baseline detected {len(iqr_anomalies)} anomalous months:')
if len(iqr_anomalies) > 0:
    print(iqr_anomalies[['month', 'rows', 'iqr_outlier_count']].to_string(index=False))

Running Isolation Forest...

Isolation Forest detected 2 anomalous months:
  month  rows  anomaly_score
2025-12   901       0.697432
2025-10   373       0.557357

IQR baseline detected 2 anomalous months:
  month  rows  iqr_outlier_count
2025-12   901                  2
2026-02   335                  1


In [ ]:
print('\n=== Comparative Summary ==="')
comparison = anomaly_result.dataframe[['month', 'rows', 'avg_tone', 'goldstein_scale', 'num_articles', 'anomaly_score', 'is_anomaly']].copy()
comparison['iqr_is_anomaly'] = baseline_result.dataframe['iqr_is_anomaly'].values
comparison['iqr_outlier_count'] = baseline_result.dataframe['iqr_outlier_count'].values
print(comparison.sort_values('anomaly_score', ascending=False).head(15).to_string(index=False))


=== Comparative Summary ==="
  month  rows  avg_tone  goldstein_scale  num_articles  anomaly_score  is_anomaly  iqr_is_anomaly  iqr_outlier_count
2025-12   901 -2.577619         0.418091          5033       0.697432        True            True                  2
2025-10   373  0.486699         1.248257          2040       0.557357        True           False                  0
2026-02   335 -2.379476        -0.046866          1640       0.555421       False            True                  1
2025-06   209 -1.575583         0.703828          1059       0.525762       False           False                  0
2025-01   532 -1.213568         0.418985          2943       0.494681       False           False                  0
2025-07   465 -0.392100         1.015914          2204       0.454543       False           False                  0
2025-09   330 -0.505516         0.745152          1557       0.447708       False           False                  0
2025-05   426 -0.019950         0.

## Summary
- **BERTopic**: Extracts semantic topics from event labels (e.g., security, diplomacy, governance).
- **NER**: Identifies key persons, organizations, and locations from GDELT structured fields.
- **Isolation Forest**: Flags months with anomalous event patterns (volume, sentiment, conflict intensity).